# PWM Example 1 — Edge-aligned PWM and duty cycle

**Target:** STM32F103C8T6 · TIM2_CH1 / PA0  
**Focus:** Configure a continuous hardware waveform with a selectable duty cycle.

## Feature classification

| Setting | Variants | Effect |
|---|---|---|
| Counter alignment | Edge-aligned up; center-aligned up/down | Changes period formula and switching symmetry |
| Output compare | PWM1 / PWM2 | Defines active interval relative to CNT and CCR |
| Polarity | High / low | Inverts the active output level |
| Duty | CCR versus ARR | Fraction of the period in the active state |
| Start | HAL start / complementary start | Enables channel output (and MOE for advanced timer as required) |

## Worked calculation — 1 kHz PWM with 25% duty

With a 72 MHz TIM2 clock, prescaler 71 yields a 1 MHz counter clock. ARR=999 gives 1 kHz edge-aligned PWM. CCR1=250 produces approximately 25% active time in PWM mode 1, active-high. For the F103 timer convention, HAL PSC and ARR are the values loaded into the registers; the frequency formula uses each value plus one.

## Compare against the repository project

`STM32F103C8T6_TIM_PWM_Output` uses TIM2 PSC=139, ARR=255 and PA0–PA2. At 72 MHz, `f_PWM = 72 MHz / (140 × 256) ≈ 2008.93 Hz`. Its RGB application updates CCR values from 0 to 255. Recalculate before changing PSC/ARR: frequency and duty resolution are linked.

CubeMX: enable TIM2_CH1 PWM on PA0, choose PWM mode 1, high polarity, PSC=71, counter period=999, pulse=250. PA0 cannot simultaneously be used as ADC1_IN0 or a button input.

```c
HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);

// For ARR = 999, 25% duty corresponds to CCR = 250.
__HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, 250U);
```


## Frequency and duty formulas

- Edge aligned: `f_PWM = f_TIM / ((PSC + 1) × (ARR + 1))`.
- Approximate active-high PWM1 duty: `CCR / (ARR + 1)` for `0 ≤ CCR ≤ ARR+1`.
- To set percentage `d`: `CCR = d × (ARR + 1) / 100`, with integer rounding and range checks.

At 1 MHz counter clock and ARR=999, one count is 1 µs and the period is 1 ms. The timer is 16-bit; increasing frequency reduces available duty resolution.

## Application variations

- **LED dimming:** a few hundred Hz or higher avoids visible flicker; verify LED polarity and current-limiting resistor.
- **Servo command:** choose the required pulse period and pulse width from the specific servo documentation; do not assume all servos share an identical range.
- **Tone:** use a square-wave/PWM output and select audible frequency; a speaker may require a driver.
- **Motor:** use a suitable power driver; MCU pins cannot drive a motor directly.

PWM is a digital output, not a true analog voltage. Filtering can recover an average only when the load and ripple requirements permit.